# GraphRAG, One Tiny Story at a Time
### A beginner-friendly walkthrough of *our actual* `refactor/v4-deep-agent` code

**Goal:** understand the path from plain text → chunks → LLM-extracted entities and relationships → Neo4j graph → Cypher traversal → answer.

This is a learning notebook, not a benchmark and not a replacement for the app. It calls the project's real functions where practical, then lets us inspect each stage.

> **Before running:** open this notebook from the repository root, activate the project's `.venv`, and make sure `.env` has working LLM and Neo4j settings. Graph extraction makes real LLM calls and writes to Neo4j. Use a dedicated/dev database if possible.

## 0. The whole story in one picture

1. **Story text** — our raw information.
2. **Chunking** — split the story into smaller text pieces.
3. **LLMGraphTransformer** — read each chunk and propose structured entities + relationships.
4. **`convert_to_graph_documents()`** — return the extracted structure as `GraphDocument` objects.
5. **`add_graph_documents()`** — write those graph documents into Neo4j.
6. **Cypher / local search** — find an entity and follow its connected edges.
7. **LLM answer** — explain the retrieved graph facts in ordinary language.

Our project also has a separate vector path: the same chunks can be embedded and stored in Milvus. This notebook focuses on the **Neo4j graph path** so we can see the graph clearly.

## 1. Our tiny story

We use a small, deliberately connected story. Each sentence adds a fact that can become a node or an edge.

- Mia lives in Willow Village.
- Mia grows vegetables in a community garden.
- Omar teaches Mia how to collect rainwater.
- Mia uses the rainwater to water the garden.
- The garden provides vegetables to the village school.
- The school cook, Lina, prepares lunch using those vegetables.

**Think first:** if asked “How does Omar help the school?”, the answer is not stated in one sentence. We need to connect Omar → Mia → rainwater/garden → vegetables → school. This is the kind of connected-fact question a graph can help retrieve—provided the LLM extracts those facts correctly.

In [ ]:
story = """
Mia lives in Willow Village. Mia grows vegetables in a community garden.
Omar teaches Mia how to collect rainwater. Mia uses the rainwater to water the garden.
The garden provides vegetables to the village school. Lina is the school cook, and Lina prepares lunch using those vegetables.
""".strip()

print(story)
print("\nCharacters:", len(story))

## 2. Connect to the real project code

The notebook imports the functions in this branch:

- `src.ingestion.ingest.chunk_documents`
- `src.graph.graph_builder.transformer`
- `src.graph.graph_builder.build_graph_from_chunks`
- `src.storage.neo4j_client.get_driver`
- `src.retrieval.graph.local_search` (later, for the real retrieval function)

If an import fails, check that the notebook kernel is using the repository's `.venv` and that you launched Jupyter from the repository root.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError(
        "Please start Jupyter from the graphrag repository root. "
        f"Current directory: {PROJECT_ROOT}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from langchain_core.documents import Document
from src.ingestion.ingest import chunk_documents
from src.graph.graph_builder import transformer, build_graph_from_chunks
from src.storage.neo4j_client import get_driver

print("Project root:", PROJECT_ROOT)
print("Imports successful")

## 3. Chunking: why split the story?

An LLM usually works more reliably when we process manageable pieces rather than one huge document. Chunking creates smaller `Document` objects; each has:

- `page_content`: the text in this chunk
- `metadata`: extra information such as page/source

The production ingestion function defaults to `chunk_size=800` and `chunk_overlap=100`. For this tiny story, we use smaller values so the pieces are easy to inspect. **This is a teaching setting, not a recommended production setting.**

In [ ]:
source_doc = Document(
    page_content=story,
    metadata={"source": "tiny_garden_story", "page": 1}
)

chunks = chunk_documents([source_doc], chunk_size=180, chunk_overlap=30)

print("Number of chunks:", len(chunks))
for i, chunk in enumerate(chunks, start=1):
    print(f"\\n--- CHUNK {i} ---")
    print("Metadata:", chunk.metadata)
    print("Text:", chunk.page_content)

### What should I notice?

- Chunking **does not** create a knowledge graph. It only splits text.
- Overlap repeats a little text between neighboring chunks so a fact near a boundary is less likely to be separated from its context.
- The actual project ingestion code first processes supported files (PDF/DOCX/PPTX/XLSX) and then calls `chunk_documents`. Here, we create a `Document` directly to keep the lesson focused.

## 4. What does `LLMGraphTransformer` do?

Imagine asking an assistant to read one chunk and write down:

- **Entities:** people, places, things, concepts
- **Relationships:** how those things are connected
- **Properties:** extra details, when extracted

For a sentence like “Omar teaches Mia how to collect rainwater”, a plausible extraction is:

`Omar` — **TEACHES** → `Mia`

The exact labels, names, and properties are proposed by the LLM; they are **not guaranteed** to be identical every run. The transformer is not simply splitting words, and it is not the database. It converts natural language into structured graph data.

In [ ]:
# Preview extraction for ONE chunk before saving anything to Neo4j.
# This is a real LLM call, using the model configured by the project.
preview_graph_documents = transformer.convert_to_graph_documents([chunks[0]])

print("Returned type:", type(preview_graph_documents))
print("Number of GraphDocument objects:", len(preview_graph_documents))

for doc_i, graph_doc in enumerate(preview_graph_documents, start=1):
    print(f"\\n=== GraphDocument {doc_i} ===")
    print("Source document:", graph_doc.source)
    print("Nodes:")
    for node in graph_doc.nodes:
        print({"id": node.id, "type": node.type, "properties": node.properties})
    print("Relationships:")
    for rel in graph_doc.relationships:
        print({
            "source": rel.source.id,
            "type": rel.type,
            "target": rel.target.id,
            "properties": rel.properties
        })

### Input vs output

| Stage | Input | Output |
|---|---|---|
| `convert_to_graph_documents([chunk])` | A list containing a LangChain `Document` with text + metadata | A list of `GraphDocument` objects, each containing nodes, relationships, and source information |

**Important:** extraction can miss a fact, produce an unexpected relation name, or return an empty graph. That is why we inspect the output instead of assuming it is perfect.

## 5. What does `build_graph_from_chunks()` do?

In this branch, `src/graph/graph_builder.py` loops over the chunks. For each chunk it:

1. calls `transformer.convert_to_graph_documents([chunk])`;
2. adds `document_id` to each extracted node and relationship (when provided);
3. counts the extracted nodes and relationships;
4. treats an empty graph as a failed chunk;
5. calls `graph.add_graph_documents(graph_documents)` to write the graph into Neo4j;
6. records failed chunks and returns them.

The `batch_size` and `delay_seconds` parameters add a pause after each batch to reduce rate-limit pressure. **This function writes to the connected Neo4j database.**

In [ ]:
# Use a unique-ish ID for this small teaching dataset.
# Re-running can create/update graph data according to Neo4jGraph's write behavior;
# use a dedicated development database and inspect before cleaning anything.
document_id = "notebook_tiny_garden_story_v1"

failed_chunks = build_graph_from_chunks(
    chunks,
    batch_size=5,
    delay_seconds=1.0,
    document_id=document_id,
)

print("Failed chunks:", len(failed_chunks))
for index, failed_chunk, reason in failed_chunks:
    print("Failed chunk index:", index)
    print("Reason:", reason)
    print("Text:", failed_chunk.page_content[:300])

### What exactly does `add_graph_documents()` do?

`convert_to_graph_documents()` creates Python objects in memory. It does **not** save them to Neo4j.

`add_graph_documents()` is the write step: the LangChain `Neo4jGraph` wrapper sends the extracted nodes and relationships to Neo4j. After that, Cypher queries can inspect the stored graph.

A simple mental model:

- `convert_to_graph_documents` = “turn the notes into a structured graph-shaped package”
- `add_graph_documents` = “save that package into the graph database”

The project attaches `document_id` to extracted nodes/relationships so the data can be traced to a document and can be targeted by the existing deletion helper. Do not run deletion on a shared database without confirming what will be removed.

## 6. Inspect what is actually in Neo4j

First, count nodes and relationships carrying our `document_id`. Then list the entities and edges. These are **read-only Cypher queries**.

In [ ]:
driver = get_driver()

with driver.session() as session:
    counts = session.run(
        """
        MATCH (n)
        WHERE n.document_id = $document_id
        WITH count(n) AS node_count
        MATCH ()-[r]->()
        WHERE r.document_id = $document_id
        RETURN node_count, count(r) AS relationship_count
        """,
        document_id=document_id,
    ).single()

print("Nodes tagged with this document_id:", counts["node_count"] if counts else 0)
print("Relationships tagged with this document_id:", counts["relationship_count"] if counts else 0)

In [ ]:
with driver.session() as session:
    rows = list(session.run(
        """
        MATCH (n)
        WHERE n.document_id = $document_id
        RETURN labels(n) AS labels, n.id AS id, properties(n) AS properties
        ORDER BY id
        """,
        document_id=document_id,
    ))

for row in rows:
    print({"labels": row["labels"], "id": row["id"], "properties": row["properties"]})

In [ ]:
with driver.session() as session:
    rows = list(session.run(
        """
        MATCH (a)-[r]->(b)
        WHERE r.document_id = $document_id
        RETURN a.id AS source, type(r) AS relationship,
               b.id AS target, properties(r) AS properties
        ORDER BY source, relationship, target
        """,
        document_id=document_id,
    ))

for row in rows:
    print(f"{row['source']} --[{row['relationship']}]--> {row['target']}")
    if row["properties"]:
        print("  relationship properties:", row["properties"])

### A graph is more than a list

A text chunk is a paragraph. A graph represents entities as **nodes** and connections as **relationships (edges)**. For example, if the extraction produces it, the database may represent a connection like:

`Omar` — `TEACHES` → `Mia` — `GROWS` → `Vegetables` — `PROVIDED_TO` → `School`

The real relationship names may differ because the LLM chooses them. Always inspect the actual output above before relying on an example edge name.

## 7. Cypher: ask Neo4j for connected facts

Cypher is Neo4j's query language. In a pattern such as:

`MATCH (a)-[r*1..2]-(b)`

- `(a)` and `(b)` are nodes;
- `-[r*1..2]-` means follow between **1 and 2 relationships**;
- the pattern is undirected here, so traversal may follow relationships either way;
- `WHERE` filters matches;
- `RETURN` chooses what to show.

First, look at all paths of 1–3 relationships that touch a node whose `id` contains “Mia”. This is a read-only exploration query.

In [ ]:
with driver.session() as session:
    rows = list(session.run(
        """
        MATCH path = (a)-[rels*1..3]-(b)
        WHERE a.document_id = $document_id
          AND any(n IN nodes(path) WHERE toLower(coalesce(n.id, '')) CONTAINS 'mia')
          AND all(n IN nodes(path) WHERE n.document_id = $document_id)
        RETURN [n IN nodes(path) | n.id] AS node_path,
               [r IN relationships(path) | type(r)] AS relationship_path
        LIMIT 25
        """,
        document_id=document_id,
    ))

for row in rows:
    print("Nodes:", row["node_path"])
    print("Edges:", row["relationship_path"])

## 8. What is “hop” or “depth”?

A **hop** is one relationship step from one node to another.

- **1 hop:** direct neighbors only.
- **2 hops:** neighbors of neighbors.
- **3 hops:** can connect facts through up to three edges.

Our actual `local_search()` in `src/retrieval/graph.py` uses two different traversal ranges:
- for one entity: `MATCH (n)-[r*1..2]-(neighbor)` → up to **2 hops**, `LIMIT 20`;
- when the query extractor returns at least two entities: `MATCH path = (a)-[r*1..3]-(b)` → paths of **1–3 hops**, `LIMIT 10`.

Those values are written directly in the Cypher patterns in this branch; they are not a single global “depth” setting. More hops can find longer connections, but can also return more irrelevant paths.

## 9. Try the project's real `local_search()` function

This is the same retrieval function the branch uses. It:
1. takes an entity name (or list of names) and the question;
2. runs Cypher against Neo4j to collect connected facts;
3. if a two-entity path is requested, tries to find a path up to 3 hops and then independently searches each entity too;
4. formats the found graph facts as text;
5. sends those facts + the original question to the configured LLM for a natural-language answer.

Because LLM extraction is not deterministic, use the node/relationship inspection above to choose names that actually exist in your graph.

In [ ]:
from src.retrieval.graph import local_search, extract_entities

question = "How does Omar help the village school through Mia?"

# First inspect what the configured LLM extracts from the question.
entities = extract_entities(question)
print("Question:", question)
print("Extracted entities:", entities)

# Now call the actual retrieval function with the extracted entities.
answer = local_search(entities, question)
print("\\nAnswer:\\n", answer)

### Try these questions

Run the cell above again after changing `question`. Suggested experiments:

1. `What does Mia do in the garden?` — mostly direct entity facts.
2. `How does Omar help the village school through Mia?` — requires a connected chain.
3. `How is Lina connected to Mia?` — tests whether the extracted graph links their facts.
4. `What connects Willow Village and the school?` — tests a longer path, if the graph contains it.
5. `What does the story say about a spaceship?` — a negative test: the graph should not invent stored facts; it may return no graph data.

Remember: the entity extractor may return different entity names than you expect. Check its printed output and compare it to the stored `n.id` values.

## 10. The other functions: `smart_query()` and `global_search()`

- `smart_query(question)` is the routing entry point. `classify_query()` decides whether the question is `LOCAL` or `GLOBAL`. Local questions use entity extraction + `local_search`; global questions use `global_search`.
- `global_search(question)` in this branch reads `(:Community {community_id, summary})` nodes, asks the LLM whether each community summary helps answer the question, and combines the partial answers.

**Why we are not testing global search on this tiny story:** the notebook only builds the entity/relationship graph. It does not run the project's community-detection and summary-generation steps, so there may be no `Community` nodes to search. A tiny graph is ideal for understanding local traversal; a larger corpus is better for meaningful global summaries.

## 11. What about Milvus and embeddings?

The full application has a parallel vector path. In `src/ingestion/ingest.py`, `embed_and_ingest()`:

1. takes the chunks' `page_content`;
2. creates embeddings using the project's configured embedder;
3. inserts chunk text, embedding, IDs, filename/source metadata, and page number into Milvus.

That vector route supports semantic similarity search. The Neo4j route extracts and stores entities and relationships. In a hybrid setup, both can complement each other: vector search can find text relevant in meaning, while graph traversal can connect explicit relationships.

This notebook intentionally skips inserting the story into Milvus so the core graph concepts remain simple.

## 12. Beginner-level checks (unit-test mindset)

These checks verify the parts that can be tested without assuming the LLM will always extract exactly the same graph. The transformer output is intentionally inspected rather than compared against one rigid expected answer.

In [ ]:
# Check 1: story is non-empty and chunking returned content.
assert story
assert chunks
assert all(chunk.page_content.strip() for chunk in chunks)
print("PASS: story and chunking")

# Check 2: the transformer should return a list-like collection.
# (The extraction itself is probabilistic, so do not assert exact node names here.)
assert isinstance(preview_graph_documents, list)
assert all(hasattr(gd, "nodes") and hasattr(gd, "relationships") for gd in preview_graph_documents)
print("PASS: transformer returned GraphDocument-shaped objects")

# Check 3: query result containers were read successfully.
assert isinstance(rows, list)
print("PASS: Neo4j query returned a list of rows (it may be empty if extraction/write failed)")

## 13. Final mental model

**Ingestion:** document → text → chunks  
**Graph building:** each chunk → LLMGraphTransformer → `GraphDocument` → `add_graph_documents()` → Neo4j  
**Graph retrieval:** question → entity extraction → Cypher traversal (1–2 hops for one entity; 1–3 hops for a pair) → retrieved graph facts  
**Answer:** retrieved facts + question → LLM → natural-language response

### What GraphRAG is good at

It can help answer questions where facts are spread across connected entities or documents. But it is not automatically accurate: extraction may miss or mislabel relationships, retrieval can miss a path, and the final LLM can still make mistakes. Inspect the graph and the retrieved facts to understand why an answer was produced.

### Cleanup note

This notebook deliberately does **not** delete anything automatically. If you used a dedicated database and want to remove the test graph, first verify what your project's `delete_graph_document(document_id)` function deletes. The current graph builder tags extracted nodes and relationships with `document_id`; use cleanup only after confirming the scope and that the database is not shared.